## How to access STAC Collections from the private STAC API at terrabyte - using DESIS L2A as example

### Requirements:

1. You need access rights to the DSS container with the DESIS data (pn56su-dss-0033).
    - Load terrapi with `module use /dss/dsstbyfs01/pn56su/pn56su-dss-0020/usr/share/modules/files/; module load terrapi` and than list all available Container and request access for the DESIS Container `terrapi restricted-data list-available` and `terrapi restricted-data request-access <NAME>`
2. Before you start querying the private STAC API with the [`terrapi-client`](https://github.com/DLR-terrabyte/terrapi-client) lib, you need to authentificate yourself with the [terrapi cli](https://docs.terrabyte.lrz.de/software/tools/terrapi/usage/terrapi/stac/auth/): ```terrapi stac auth```

-> `pip install git+https://github.com/DLR-terrabyte/terrapi-client.git@latest-release` into your python env.

-> `pip install git+https://github.com/DLR-terrabyte/terrapi.git@latest-release`

In [1]:
import pystac
from odc import stac as odc_stac
import dask
import time
import xarray
from dask.distributed import Client
import folium
import folium.plugins as folium_plugins
import geopandas as gpd
from odc.geo import geobox
from terrapi.client.stac import open_private_catalog

In [4]:
# Connect to the terrabyte private STAC API
catalog = open_private_catalog()

date = ["2020-05-01", "2020-09-30"]
query_1 = {
    'eo:cloud_cover': {
        "gte": 0,
        "lte": 20
    }
}

query_2 = {
    'eo:cloud_cover': {
        "gte": 20
    }
}

results = catalog.search(
    collections=[f"pn56su-dss-0033.DESIS_HSI_L2A"],
    datetime=date
)
items = results.item_collection_as_dict()

results_cloud_cover_1 = catalog.search(
    collections=[f"pn56su-dss-0033.DESIS_HSI_L2A"],
    datetime=date,
    query=query_1
)
items_cloud_cover_1 = results_cloud_cover_1.item_collection_as_dict()

results_cloud_cover_2 = catalog.search(
    collections=[f"pn56su-dss-0033.DESIS_HSI_L2A"],
    datetime=date,
    query=query_2
)
items_cloud_cover_2 = results_cloud_cover_2.item_collection_as_dict()

print(f"In total {len(items['features'])} items found for daterange {date}. For a cloud cover not higher then 20 there are {len(items_cloud_cover_1['features'])} items and {len(items_cloud_cover_2['features'])} for a cloud cover higher then 20."  )

In total 1950 items found for daterange ['2020-05-01', '2020-09-30']. For a cloud cover not higher then 20 there are 1567 items and 383 for a cloud cover higher then 20.


## Inspect STAC Items

In [6]:
items_cloud_cover = list(results_cloud_cover_1.items())

In [7]:
items_cloud_cover_2 = list(results_cloud_cover_2.items())

In [8]:
items_cloud_cover[10]

<Item id=DESIS-HSI-L2A-DT0503815508_004-20200929T123405-V0220>

In [9]:
from odc import stac as odc_stac

desis_datacube = odc_stac.load([items_cloud_cover[0]], bands = ['b1', 'b2', 'b3'])
desis_datacube

<xarray.Dataset> Size: 13MB
Dimensions:      (y: 1450, x: 1487, time: 1)
Coordinates:
  * y            (y) float64 12kB 1.184e+06 1.184e+06 ... 1.141e+06 1.141e+06
  * x            (x) float64 12kB 6.751e+05 6.751e+05 ... 7.196e+05 7.197e+05
    spatial_ref  int32 4B 32616
  * time         (time) datetime64[ns] 8B 2020-09-30T17:56:33.277770
Data variables:
    b1           (time, y, x) int16 4MB -32768 -32768 -32768 ... -32768 -32768
    b2           (time, y, x) int16 4MB -32768 -32768 -32768 ... -32768 -32768
    b3           (time, y, x) int16 4MB -32768 -32768 -32768 ... -32768 -32768

In [13]:
map = folium.Map()
layer_control = folium.LayerControl(position='topright', collapsed=True)
fullscreen = folium_plugins.Fullscreen()
style_1 = {'fillColor': '#00000000', "color": "#0000ff", "weight": 2}
style_2 = {'fillColor': '#00000000', "color": "#ff0000", "weight": 2}

footprints_clouds_1 = folium.GeoJson(
    gpd.GeoDataFrame.from_features([item.to_dict() for item in items_cloud_cover]).to_json(),
    name='STAC Items footprints with cloud cover lower then 20 (blue)',
    style_function=lambda x: style_1,
    control=True
)

footprints_clouds_2 = folium.GeoJson(
    gpd.GeoDataFrame.from_features([item.to_dict() for item in items_cloud_cover_2]).to_json(),
    name='STAC Items footprints with cloud cover greater then 20 (red)',
    style_function=lambda x: style_2,
    control=True
)

footprints_clouds_2.add_to(map)
footprints_clouds_1.add_to(map)
layer_control.add_to(map)
fullscreen.add_to(map)
map.fit_bounds(map.get_bounds())
map